<a href="https://colab.research.google.com/github/Rennanmax80/MVP-MODULO-3/blob/main/mvp_puc_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# 1. IMPORTS
# ===============================
from pathlib import Path
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ===============================
# 2. LOAD DATASET (URL)
# ===============================
columns = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety",
    "class",
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
df = pd.read_csv(url, names=columns)

df.head()

# ===============================
# 3. FEATURES / TARGET
# ===============================
X = df.drop(columns=["class"])
y = df["class"]

# ===============================
# 4. SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = X.columns.tolist()

one_hot_preprocessor = ColumnTransformer(
    [("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)]
)

ordinal_preprocessor = ColumnTransformer(
    [("categorical", OrdinalEncoder(), categorical_features)]
)

# ===============================
# 5. MODELS + PIPELINES + TUNING
# ===============================
model_configs = {
    "KNN": {
        "pipeline": Pipeline([
            ("preprocess", one_hot_preprocessor),
            ("model", KNeighborsClassifier()),
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9],
            "model__weights": ["uniform", "distance"],
        },
    },
    "DecisionTree": {
        "pipeline": Pipeline([
            ("preprocess", ordinal_preprocessor),
            ("model", DecisionTreeClassifier(random_state=42)),
        ]),
        "params": {
            "model__max_depth": [None, 5, 10, 15],
            "model__min_samples_leaf": [1, 2, 4],
        },
    },
    "NaiveBayes": {
        "pipeline": Pipeline([
            ("preprocess", ordinal_preprocessor),
            ("model", CategoricalNB()),
        ]),
        "params": {
            "model__alpha": [0.1, 0.5, 1.0],
        },
    },
    "SVM": {
        "pipeline": Pipeline([
            ("preprocess", one_hot_preprocessor),
            ("model", SVC(probability=True, random_state=42)),
        ]),
        "params": {
            "model__C": [0.5, 1.0, 2.0],
            "model__kernel": ["linear", "rbf"],
        },
    },
}

# ===============================
# 6. TRAIN + EVALUATE
# ===============================
results = {}
best_estimators = {}

for name, config in model_configs.items():
    search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
    )
    search.fit(X_train, y_train)

    best_estimators[name] = search.best_estimator_

    y_pred = search.best_estimator_.predict(X_test)

    results[name] = {
        "cv_accuracy": search.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1_macro": f1_score(y_test, y_pred, average="macro"),
        "best_params": search.best_params_,
    }

results_df = pd.DataFrame(results).T.sort_values(
    ["test_f1_macro", "test_accuracy", "cv_accuracy"], ascending=False
)
results_df

# ===============================
# 7. BEST MODEL
# ===============================
best_model_name = results_df.index[0]
best_model = best_estimators[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model:", best_model_name)
print("Metrics:", results[best_model_name])
print(classification_report(y_test, best_predictions))

# ===============================
# 8. SAVE MODEL
# ===============================
backend_dir = Path("..") / "backend"
if backend_dir.exists():
    model_output_path = backend_dir / "model.pkl"
else:
    model_output_path = Path("model.pkl")

model_output_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_output_path)
print(f"Model saved to: {model_output_path.resolve()}")

# ===============================
# 9. QUICK PREDICTION CHECK
# ===============================
sample_df = pd.DataFrame([
    {
        "buying": "low",
        "maint": "low",
        "doors": "4",
        "persons": "4",
        "lug_boot": "big",
        "safety": "high",
    }
])
sample_prediction = best_model.predict(sample_df)[0]
sample_confidence = float(best_model.predict_proba(sample_df).max())

print(f"Sample class: {sample_prediction}")
print(f"Sample confidence: {sample_confidence:.4f}")

Best model: SVM
Metrics: {'cv_accuracy': np.float64(0.9869800659237168), 'test_accuracy': 0.9942196531791907, 'test_f1_macro': 0.987614958754389, 'best_params': {'model__C': 2.0, 'model__kernel': 'rbf'}}
              precision    recall  f1-score   support

         acc       0.99      0.99      0.99        77
        good       0.93      1.00      0.97        14
       unacc       1.00      1.00      1.00       242
       vgood       1.00      1.00      1.00        13

    accuracy                           0.99       346
   macro avg       0.98      1.00      0.99       346
weighted avg       0.99      0.99      0.99       346

Model saved to: /content/model.pkl
Sample class: vgood
Sample confidence: 0.9864
